In [0]:
%sql
CREATE CATALOG muhib_nxvora;

In [0]:
from pyspark.sql import Row
from pyspark.sql.functions import col, datediff, to_date, when, current_date, lit

# 25 rows — simulates the morning CSV dump
raw_data = [
    ("ORD001", "Alice",   "Electronics", "2024-03-01", "2024-03-04", "Delivered"),
    ("ORD002", "Bob",     "Clothing",    "2024-03-01", "2024-03-08", "Delivered"),
    ("ORD003", "Carol",   "Groceries",   "2024-03-02", "2024-03-03", "Delivered"),
    ("ORD004", "David",   "Electronics", "2024-03-02", "2024-03-10", "Delivered"),
    ("ORD005", "Eva",     "Home",        "2024-03-03", "2024-03-05", "Delivered"),
    ("ORD006", "Frank",   "Clothing",    "2024-03-03", "2024-03-12", "Delivered"),
    ("ORD007", "Grace",   "Groceries",   "2024-03-04", "2024-03-05", "Delivered"),
    ("ORD008", "Henry",   "Electronics", "2024-03-04", "2024-03-14", "Delivered"),
    ("ORD009", "Isla",    "Home",        "2024-03-05", "2024-03-07", "Delivered"),
    ("ORD010", "Jack",    "Clothing",    "2024-03-05", "2024-03-13", "Delivered"),
    ("ORD011", "Karen",   "Groceries",   "2024-03-06", "2024-03-07", "Delivered"),
    ("ORD012", "Leo",     "Electronics", "2024-03-06", "2024-03-09", "Delivered"),
    ("ORD013", "Mia",     "Home",        "2024-03-07", "2024-03-16", "Delivered"),
    ("ORD014", "Nolan",   "Clothing",    "2024-03-07", "2024-03-09", "Delivered"),
    ("ORD015", "Olivia",  "Groceries",   "2024-03-08", "2024-03-09", "Delivered"),
    ("ORD016", "Paul",    "Electronics", "2024-03-08", "2024-03-17", "Delivered"),
    ("ORD017", "Quinn",   "Home",        "2024-03-09", "2024-03-11", "Delivered"),
    ("ORD018", "Rachel",  "Clothing",    "2024-03-09", "2024-03-10", "Delivered"),
    ("ORD019", "Sam",     "Groceries",   "2024-03-10", "2024-03-11", "Delivered"),
    ("ORD020", "Tina",    "Electronics", "2024-03-10", "2024-03-19", "Delivered"),
    ("ORD021", "Uma",     "Home",        "2024-03-11", "2024-03-13", "Delivered"),
    ("ORD022", "Victor",  "Clothing",    "2024-03-11", "2024-03-20", "Delivered"),
    ("ORD023", "Wendy",   "Groceries",   "2024-03-12", "2024-03-13", "Delivered"),
    ("ORD024", "Xander",  "Electronics", "2024-03-12", "2024-03-14", "Delivered"),
    ("ORD025", "Yara",    "Home",        "2024-03-13", "2024-03-22", "Delivered"),
]

columns = ["order_id", "customer", "category", "order_date", "delivery_date", "status"]

df_raw = spark.createDataFrame(raw_data, columns)
print(f"Total orders received: {df_raw.count()}")
df_raw.show(truncate=False)

Total orders received: 25
+--------+--------+-----------+----------+-------------+---------+
|order_id|customer|category   |order_date|delivery_date|status   |
+--------+--------+-----------+----------+-------------+---------+
|ORD001  |Alice   |Electronics|2024-03-01|2024-03-04   |Delivered|
|ORD002  |Bob     |Clothing   |2024-03-01|2024-03-08   |Delivered|
|ORD003  |Carol   |Groceries  |2024-03-02|2024-03-03   |Delivered|
|ORD004  |David   |Electronics|2024-03-02|2024-03-10   |Delivered|
|ORD005  |Eva     |Home       |2024-03-03|2024-03-05   |Delivered|
|ORD006  |Frank   |Clothing   |2024-03-03|2024-03-12   |Delivered|
|ORD007  |Grace   |Groceries  |2024-03-04|2024-03-05   |Delivered|
|ORD008  |Henry   |Electronics|2024-03-04|2024-03-14   |Delivered|
|ORD009  |Isla    |Home       |2024-03-05|2024-03-07   |Delivered|
|ORD010  |Jack    |Clothing   |2024-03-05|2024-03-13   |Delivered|
|ORD011  |Karen   |Groceries  |2024-03-06|2024-03-07   |Delivered|
|ORD012  |Leo     |Electronics|2024-

In [0]:
df_flagged = (
    df_raw
    .withColumn("order_date",    to_date("order_date"))
    .withColumn("delivery_date", to_date("delivery_date"))
    .withColumn("days_to_deliver",
        datediff(col("delivery_date"), col("order_date"))
    )
    .withColumn("delivery_status",
        when(col("days_to_deliver") > 5, "🔴 LATE")
        .otherwise("🟢 ON TIME")
    )
)

df_flagged.select(
    "order_id", "customer", "category",
    "days_to_deliver", "delivery_status"
).show(truncate=False)

+--------+--------+-----------+---------------+---------------+
|order_id|customer|category   |days_to_deliver|delivery_status|
+--------+--------+-----------+---------------+---------------+
|ORD001  |Alice   |Electronics|3              |🟢 ON TIME     |
|ORD002  |Bob     |Clothing   |7              |🔴 LATE        |
|ORD003  |Carol   |Groceries  |1              |🟢 ON TIME     |
|ORD004  |David   |Electronics|8              |🔴 LATE        |
|ORD005  |Eva     |Home       |2              |🟢 ON TIME     |
|ORD006  |Frank   |Clothing   |9              |🔴 LATE        |
|ORD007  |Grace   |Groceries  |1              |🟢 ON TIME     |
|ORD008  |Henry   |Electronics|10             |🔴 LATE        |
|ORD009  |Isla    |Home       |2              |🟢 ON TIME     |
|ORD010  |Jack    |Clothing   |8              |🔴 LATE        |
|ORD011  |Karen   |Groceries  |1              |🟢 ON TIME     |
|ORD012  |Leo     |Electronics|3              |🟢 ON TIME     |
|ORD013  |Mia     |Home       |9              |🔴 LAT

In [0]:
df_flagged.write.format("delta").mode("overwrite").saveAsTable("orders_flagged")

# Now query it like a table
spark.sql("""
    SELECT delivery_status, COUNT(*) AS total_orders
    FROM orders_flagged
    GROUP BY delivery_status
    ORDER BY total_orders DESC
""").show()

+---------------+------------+
|delivery_status|total_orders|
+---------------+------------+
|     🟢 ON TIME|          15|
|        🔴 LATE|          10|
+---------------+------------+



In [0]:
spark.sql("""
    SELECT
        category,
        COUNT(*)                                        AS total_orders,
        SUM(CASE WHEN delivery_status = '🔴 LATE' THEN 1 ELSE 0 END)  AS late_orders,
        ROUND(AVG(days_to_deliver), 1)                  AS avg_days,
        MAX(days_to_deliver)                            AS worst_case_days
    FROM orders_flagged
    GROUP BY category
    ORDER BY late_orders DESC
""").show()

print("Done! Operations team now has a clear report instead of scanning 25 rows manually.")

+-----------+------------+-----------+--------+---------------+
|   category|total_orders|late_orders|avg_days|worst_case_days|
+-----------+------------+-----------+--------+---------------+
|Electronics|           7|          4|     6.3|             10|
|   Clothing|           6|          4|     6.0|              9|
|       Home|           6|          2|     4.3|              9|
|  Groceries|           6|          0|     1.0|              1|
+-----------+------------+-----------+--------+---------------+

Done! Operations team now has a clear report instead of scanning 25 rows manually.
